In [1]:
import json

texts = []
total_chars = 0
target_chars = 1_000_000

with open("../data/raw/train.jsonl", encoding="utf-8") as f:
    for line in f:
        text = json.loads(line)["text"]
        texts.append(text)
        total_chars += len(text)

        if total_chars >= target_chars:
            break

print("Stories:", len(texts))
print("Characters:", total_chars)

Stories: 1222
Characters: 1000670


In [2]:
TARGET_VOCAB_SIZE = 8192

base_vocab_size = 256
special_tokens = 1

max_merges = TARGET_VOCAB_SIZE - base_vocab_size - special_tokens

print("Target vocabulary:", TARGET_VOCAB_SIZE)
print("BPE merges:", max_merges)

Target vocabulary: 8192
BPE merges: 7935


In [ ]:
import sys
sys.path.append("..")
from tokenizer.bpe_v1 import ByteLevelBPETokenizer
import time


start_time = time.time()

TEST_VOCAB_SIZE = 512

tokenizer = ByteLevelBPETokenizer()

tokenizer.initialize_vocab()
tokenizer.add_special_tokens(["<|endoftext|>"])

tokenizer.train(
    texts,
    vocab_size=TEST_VOCAB_SIZE
)

elapsed = time.time() - start_time

print(f"Training time: {elapsed:.2f} seconds")

Training time: 20.71 seconds


In [4]:
import sys
sys.path.append("..")
from tokenizer.bpe_v2 import ByteLevelBPETokenizer
import time


start_time = time.time()

TEST_VOCAB_SIZE = 512

tokenizer = ByteLevelBPETokenizer()

tokenizer.initialize_vocab()
tokenizer.add_special_tokens(["<|endoftext|>"])

tokenizer.train(
    texts,
    vocab_size=TEST_VOCAB_SIZE
)

elapsed = time.time() - start_time

print(f"Training time: {elapsed:.2f} seconds")

Training time: 35.37 seconds


## Optimization Experiment — v2

The first BPE implementation scans the full corpus again for every merge.

I tried a v2 approach where pair counts and affected sequences were tracked and updated incrementally, so we would not have to rebuild everything after every merge.

Benchmark on ~1M characters with 512 vocabulary:

- v1: 20.58 sec
- v2: 35.37 sec

v2 was actually slower.

The additional bookkeeping with `Counter`, sets, pair-to-sequence tracking, and updating statistics added more overhead than the time saved from avoiding full scans.

### Takeaway

This optimization did not work well for our current implementation and corpus size.

Keeping v1 as the simple baseline and trying a different optimization approach next.

### bpe.py experiment -- Final

In [ ]:
import sys
sys.path.append("..")
from tokenizer.bpe_v1 import ByteLevelBPETokenizer
import time
from collections import Counter


tokenizer = ByteLevelBPETokenizer()
# text = 'Hello, how are you?'
all_pieces = []

for text in texts:
    all_pieces.extend(tokenizer.pretokenize(text))

piece_counts = Counter(all_pieces)

print("Total pieces:", len(all_pieces))
print("Unique pieces:", len(piece_counts))
print("Most common pieces:", piece_counts.most_common(20))

Total pieces: 242691
Unique pieces: 4876
Most common pieces: [('.', 18971), (',', 10513), (' the', 9484), (' and', 9161), (' a', 6897), (' to', 6718), (' was', 4810), ('\n', 3820), ('"', 2867), (' They', 2389), (' it', 2375), (' ', 2310), (' The', 2086), (' He', 2082), (' said', 1957), (' with', 1939), (' day', 1888), (' in', 1788), (' She', 1785), (' her', 1764)]


In [3]:
samples = [
    "the cat is here",
    "Hello, how are you?",
    "Billy's friend said, \"Let's play!\"",
    "One day\nBilly went home."
]

for text in samples:
    print(text)
    print(tokenizer.pretokenize(text))
    print()

the cat is here
['the', ' cat', ' is', ' here']

Hello, how are you?
['Hello', ',', ' how', ' are', ' you', '?']

Billy's friend said, "Let's play!"
["Billy's", ' friend', ' said', ',', ' ', '"', "Let's", ' play', '!"']

One day
Billy went home.
['One', ' day', '\n', 'Billy', ' went', ' home', '.']



In [4]:
piece_corpus, piece_counts = tokenizer.build_piece_corpus(texts)

pair_counts = tokenizer.count_pairs_weighted(
    piece_corpus,
    piece_counts
)

In [ ]:
import sys
sys.path.append("..")
from tokenizer.bpe_v1 import ByteLevelBPETokenizer
import time


start_time = time.time()

TEST_VOCAB_SIZE = 8192

tokenizer = ByteLevelBPETokenizer()

tokenizer.initialize_vocab()
tokenizer.add_special_tokens(["<|endoftext|>"])

tokenizer.train(
    texts,
    vocab_size=TEST_VOCAB_SIZE
)

elapsed = time.time() - start_time

print(f"Vocab Size: {TEST_VOCAB_SIZE}")
print(f"Training time: {elapsed:.2f} seconds")

Vocab Size: 8192
Training time: 12.31 seconds


In [7]:
text = "play<|endoftext|>played play<|endoftext|>playing"

tokens = tokenizer.encode(text)
decoded = tokenizer.decode(tokens)

print(tokens)
print(decoded)

assert decoded == text

print("Encode/decode round trip passed")

[805, 280, 256, 805, 280, 264, 332, 256, 805, 280, 298]
play<|endoftext|>played play<|endoftext|>playing
Encode/decode round trip passed


## Save / Load Verification

The tokenizer trained on the development corpus was saved to disk and loaded
into a new tokenizer instance.

We verified that the loaded tokenizer produces the same tokenization and can
decode the tokens back to the original text.

```text
Characters:      1,000,670
Tokens:            242,691
Characters/token: 4.12
Tokens/character: 0.243

In [ ]:
from pathlib import Path
from tokenizer.bpe_v1 import ByteLevelBPETokenizer

Path("../artifacts").mkdir(exist_ok=True)

tokenizer.save("../artifacts/tinystories_bpe_8192_dev.json")

loaded_tokenizer = ByteLevelBPETokenizer()
loaded_tokenizer.load("../artifacts/tinystories_bpe_8192_dev.json")

text = "play<|endoftext|>played play<|endoftext|>playing"

tokens = loaded_tokenizer.encode(text)
decoded = loaded_tokenizer.decode(tokens)

print("Tokens:", tokens)
print("Decoded:", decoded)

assert decoded == text

print("Save/load round trip passed")

Tokens: [805, 280, 256, 805, 280, 264, 332, 256, 805, 280, 298]
Decoded: play<|endoftext|>played play<|endoftext|>playing
Save/load round trip passed


In [9]:
total_tokens = 0
total_chars = 0

for text in texts:
    tokens = loaded_tokenizer.encode(text)
    total_tokens += len(tokens)
    total_chars += len(text)

print("Characters:", total_chars)
print("Tokens:", total_tokens)
print("Characters/token:", total_chars / total_tokens)
print("Tokens/character:", total_tokens / total_chars)

Characters: 1000670
Tokens: 242691
Characters/token: 4.123226654470088
Tokens/character: 0.2425285059010463


### Final training on full input dataset

In [ ]:
import sys
sys.path.append("..")
import json
import time
from pathlib import Path

from tokenizer.bpe_v1 import ByteLevelBPETokenizer


# Load the full training corpus
texts = []

with open("../data/raw/train.jsonl", encoding="utf-8") as file:
    for line in file:
        texts.append(json.loads(line)["text"])

print("Training stories:", len(texts))
print("Training characters:", sum(len(text) for text in texts))


# Train the final tokenizer
TARGET_VOCAB_SIZE = 8192

tokenizer = ByteLevelBPETokenizer()
tokenizer.initialize_vocab()
tokenizer.add_special_tokens(["<|endoftext|>"])

start_time = time.time()

tokenizer.train(
    texts,
    vocab_size=TARGET_VOCAB_SIZE
)

elapsed = time.time() - start_time

print("Vocabulary size:", len(tokenizer.vocab))
print("BPE merges:", len(tokenizer.merges))
print(f"Training time: {elapsed:.2f} seconds")


# Save the final tokenizer
Path("artifacts").mkdir(exist_ok=True)

tokenizer.save(
    "../artifacts/tinystories_bpe_8192.json"
)

print("Tokenizer saved.")

Training stories: 25000
Training characters: 20087753
Vocabulary size: 8191
BPE merges: 7935
Training time: 50.60 seconds
Tokenizer saved.


In [ ]:
loaded_tokenizer = ByteLevelBPETokenizer()

loaded_tokenizer.load(
    "../artifacts/tinystories_bpe_8192.json")

text = "Once upon a time<|endoftext|>The little dog was happy."

tokens = loaded_tokenizer.encode(text)
decoded = loaded_tokenizer.decode(tokens)

print("Tokens:", tokens)
print("Decoded:", decoded)

assert decoded == text

print("Final tokenizer save/load test passed")

Tokens: [430, 437, 259, 398, 256, 412, 389, 464, 283, 375, 46]
Decoded: Once upon a time<|endoftext|>The little dog was happy.
Final tokenizer save/load test passed


In [ ]:
# check the train/val data statistics

# Load the full train/val corpus
texts = []

with open("../data/raw/train.jsonl", encoding="utf-8") as file:
    for line in file:
        texts.append(json.loads(line)["text"])

print("Train stories:", len(texts))
print("Train characters:", sum(len(text) for text in texts))

total_tokens = 0
total_chars = 0

for text in texts:
    tokens = tokenizer.encode(text)
    total_tokens += len(tokens)
    total_chars += len(text)

print("Characters:", total_chars)
print("Tokens:", total_tokens)
print("Characters/token:", total_chars / total_tokens)
print("Tokens/character:", total_tokens / total_chars)

Train stories: 25000
Train characters: 20087753
Characters: 20087753
Tokens: 4907617
Characters/token: 4.093178624167289
Tokens/character: 0.24430890801972724
